In [1]:
#Imports
import json
import pandas as pd
import numpy as np
import os
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import accuracy_score, log_loss
import matplotlib.pyplot as plt

In [2]:
#Parameters
Iterations = 2048 # number of iterations for Logistic Regression
random = 32 # random state for Logistic Regression
Threshold = 0.05 # Threshold for when feature should be dropped.

In [3]:
# Load Data
api_url = "https://api.github.com/repos/COMP3608-GROUP-9-PROJECT/COMP3608_PROJECT/contents/ProcessedData/processed_tournament_data.csv"
os.makedirs("processed", exist_ok=True)

response = requests.get(api_url)

if response.status_code == 200:
    file = response.json()
    download_url = file["download_url"]

    csv_data = requests.get(download_url).content
    tournament_data = pd.read_csv(io.BytesIO(csv_data))

    print("Loaded into DataFrame successfully")
else:
    raise Exception(f"Failed to access API: {response.status_code}")

TARGET = "Win"

FEATURES = [col for col in tournament_data.columns if col not in ["Win", "Season", "PointDifference", "Division", "SeedDifference","RatingDifference"]]

x = tournament_data[FEATURES]
y = tournament_data[TARGET]

groups = tournament_data["Season"]
seasons = tournament_data["Season"].unique()

gkf = GroupKFold(n_splits=len(seasons))


all_importances = []

NameError: name 'io' is not defined

In [ ]:
# Feature Selection
for season_idx, (train_index, test_index) in enumerate(gkf.split(x, y, groups)):
    holdout_season = seasons[season_idx]

    x_train, x_test = x.iloc[train_index], x.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]


    model = LogisticRegression(penalty='l1',solver='liblinear',max_iter=Iterations,random_state=random)

    model.fit(x_train, y_train)

    coef = model.coef_[0]

    importance_df = pd.DataFrame({
        "Feature": FEATURES,
        "Coefficient": coef,
        "AbsCoefficient": np.abs(coef),
        "Season": holdout_season
    })

    all_importances.append(importance_df)

df_importance = pd.concat(all_importances)

avg_importance = (
    df_importance.groupby("Feature")["AbsCoefficient"]
    .mean()
    .sort_values(ascending=False)
)

selected_features = avg_importance[avg_importance > Threshold]

print("\nSelected Features:")
print(selected_features)



NameError: name 'df_features' is not defined

In [ ]:
os.makedirs("../ProcessedData", exist_ok=True)

selected_features.to_csv("../ProcessedData/selected_features.csv", header=True, index_label="Feature")

with open("../ProcessedData/selected_features.txt", "w") as f:
    for feature in selected_features.index:
        f.write(f"{feature}\n")

print(f"\nSaved {len(selected_features)} selected features to ProcessedData/")
print("Files created:")
print("  - selected_features.csv (importances as CSV)")
print("  - selected_features.txt (feature names only)")
